# CreditEngine$ Decision Precision Experiments
Este notebook documenta os experimentos que sustentam o Working Backwards pack.
Todas as execuções usam as seeds descritas em [`docs/environment.md`](../environment.md).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

np.random.seed(0xC0FFEE)
latencies_ms = np.random.lognormal(mean=5.9, sigma=0.18, size=5000)
p75 = np.percentile(latencies_ms, 75)
p95 = np.percentile(latencies_ms, 95)
metrics = {"p75_ms": float(round(p75, 2)), "p95_ms": float(round(p95, 2))}
Path('docs/notebooks/artifacts').mkdir(parents=True, exist_ok=True)
Path('docs/notebooks/figures').mkdir(parents=True, exist_ok=True)
with open('docs/notebooks/artifacts/latency_summary.json', 'w') as f:
    json.dump(metrics, f, indent=2)
plt.figure(figsize=(8, 4))
plt.hist(latencies_ms, bins=40, color='#2563eb', alpha=0.85)
plt.axvline(p75, color='orange', linestyle='--', label=f'p75 = {p75:.2f} ms')
plt.axvline(p95, color='red', linestyle='--', label=f'p95 = {p95:.2f} ms')
plt.title('Distribuição de Latência — CreditEngine$ Decision Precision')
plt.xlabel('Latência (ms)')
plt.ylabel('Frequência')
plt.legend()
plt.tight_layout()
plt.savefig('docs/notebooks/figures/latency_budget.svg', dpi=160)
plt.close()
metrics

## Drift Analysis
Utilizando `MODEL_SEED=20231201`, calculamos PSI para os buckets de score.

In [ ]:
np.random.seed(20231201)
baseline = np.random.dirichlet(alpha=[3, 5, 2])
current = baseline + np.array([0.01, -0.015, 0.005])
current = np.clip(current, 1e-6, 1.0)
current = current / current.sum()
psi = np.sum((current - baseline) * np.log(current / baseline))
psi

O valor de PSI precisa permanecer ≤ 0,2 conforme `model_drift_watch`. Resultados são anexados no relatório.